# Module 1 — Code Along: Python Core (the bank-accounts story)

Today we run a tiny **bank**. Everything we learn today operates on one kind of thing: an **account**.

For now an account is just a Python **dict** — a bag of named fields. (Classes arrive in Module 3; files in Module 2.)

Canonical account shape — the same six fields we reuse all day, and in M2/M3:

```python
id: int, owner: str, account_type: str, balance: float, is_active: bool, tags: list[str]
```

Run top to bottom. Every cell is self-contained and re-runnable.

## Variables & Python's types

A variable is just a name pointing at a value — no type declaration, Python infers it from the value. The account fields cover Python's core types: `str`, `int`, `float`, `bool`, `list`, `dict`, and `None`.

In [ ]:
# An account is a dict literal: named fields, no class yet. This is our whole domain today.
acct = {
    "id": 1,
    "owner": "Ada",
    "account_type": "savings",
    "balance": 1500.0,
    "is_active": True,
    "tags": ["primary", "online"],
}

# Read a field by its key. No types were declared — Python knew them from the values.
print(acct["owner"], "holds a", acct["account_type"], "account")  # Ada holds a savings account

# type() reveals the type Python inferred for each field. These ARE Python's core types.
print(type(acct["id"]))         # <class 'int'>   — whole number
print(type(acct["owner"]))      # <class 'str'>   — text
print(type(acct["balance"]))    # <class 'float'> — decimal number
print(type(acct["is_active"]))  # <class 'bool'>  — True / False
print(type(acct["tags"]))       # <class 'list'>  — ordered collection
print(type(acct))               # <class 'dict'>  — the account itself: named fields

# None is Python's 'no value' (like null). An account might have no overdraft limit set yet:
overdraft_limit = None
print(type(overdraft_limit))    # <class 'NoneType'>

## Truthiness — empty things are False

You can put a value straight into an `if`. "Empty" or "zero" values are **falsy**: `0`, `0.0`, `""`, `[]`, `{}`, `None`, `False`. Everything else is truthy. This lets us write `if acct["tags"]` instead of `if len(acct["tags"]) > 0`.

In [ ]:
acct = {"id": 1, "owner": "Ada", "account_type": "savings",
        "balance": 1500.0, "is_active": True, "tags": ["primary", "online"]}

# A freshly opened account: not active yet, zero balance, no tags — to see falsy values in action.
new_acct = {"id": 2, "owner": "Lin", "account_type": "checking",
            "balance": 0.0, "is_active": False, "tags": []}

# Empty list is falsy, so `not new_acct["tags"]` is True. WHY: cleaner than len(...) == 0.
if not new_acct["tags"]:
    print(new_acct["owner"], "has no tags yet")        # Lin has no tags yet

# is_active is already a bool, so test it directly — no `== True` needed.
if not new_acct["is_active"]:
    print(new_acct["owner"], "account is not active")  # Lin account is not active

# balance of 0.0 is falsy; Ada's 1500.0 is truthy.
if not new_acct["balance"]:
    print(new_acct["owner"], "has an empty balance")   # Lin has an empty balance
if acct["balance"]:
    print(acct["owner"], "has funds:", acct["balance"]) # Ada has funds: 1500.0

# Every falsy value, confirmed via bool():
print([bool(x) for x in (0, 0.0, "", [], {}, None, False)])  # all False

## Control flow — if / elif / else, for, while

`if/elif/else` picks one branch; `for` walks a sequence; `while` repeats until its condition goes false. Python uses **indentation** (not braces) to mark each block.

In [ ]:
# A small set of accounts to loop over.
accounts = [
    {"id": 1, "owner": "Ada", "balance": 1500.0, "is_active": True},
    {"id": 2, "owner": "Lin", "balance": 0.0,    "is_active": False},
    {"id": 3, "owner": "Sam", "balance": 40.0,   "is_active": True},
]

# for: do the same work for every account. if/elif/else: branch on active + balance band.
for a in accounts:
    if not a["is_active"]:
        label = "inactive"
    elif a["balance"] < 100:
        label = "low balance"
    else:
        label = "healthy"
    print(a["owner"], "->", label)
# Ada -> healthy / Lin -> inactive / Sam -> low balance

# while: keep going until a condition is false. Here: deposit $25 until Sam clears the $100 line.
sam = accounts[2]
while sam["balance"] < 100:
    sam["balance"] += 25
print("Sam's balance after top-ups:", sam["balance"])  # Sam's balance after top-ups: 115.0

## Functions — args, defaults, return

`def` names reusable logic. Parameters can have **defaults** (`only_active=True`); callers pass arguments by **position** or by **keyword**; `return` hands a value back to the caller.

In [ ]:
accounts = [
    {"id": 1, "owner": "Ada", "balance": 1500.0, "is_active": True},
    {"id": 2, "owner": "Lin", "balance": 200.0,  "is_active": False},
    {"id": 3, "owner": "Sam", "balance": 40.0,   "is_active": True},
]

# `only_active` has a default, so callers can omit it. WHY defaults: the common case stays short.
def total_balance(accounts, only_active=True):
    if only_active:
        accounts = [a for a in accounts if a["is_active"]]  # keep only active accounts
    return sum(a["balance"] for a in accounts)              # return the total to the caller

# Positional arg only — only_active falls back to its default True (Ada 1500 + Sam 40).
print(total_balance(accounts))                   # 1540.0

# Keyword arg — pass only_active by name; clearer at the call site, and counts Lin too.
print(total_balance(accounts, only_active=False)) # 1740.0

# A second function: positional `acct` and `amount`. Returns the updated account.
def withdraw(acct, amount):
    acct["balance"] -= amount   # mutate the balance and hand the account back
    return acct

ada = accounts[0]
print(withdraw(ada, 500)["balance"])             # 1000.0

## Exceptions — try / except / else / finally

When something goes wrong you `raise` an error instead of returning a bad value. The caller wraps risky code in `try`, handles the failure in `except`, runs `else` only on success, and `finally` always (cleanup). Here: an over-limit withdrawal, and looking up an account id that doesn't exist.

In [ ]:
accounts = [
    {"id": 1, "owner": "Ada", "balance": 1500.0, "is_active": True},
    {"id": 3, "owner": "Sam", "balance": 40.0,   "is_active": True},
]

# Raise on insufficient funds instead of letting the balance go negative.
def withdraw(acct, amount):
    if amount > acct["balance"]:
        raise ValueError(f"insufficient funds: balance {acct['balance']}, asked {amount}")
    acct["balance"] -= amount
    return acct["balance"]

# Raise a clear error when the account isn't found, instead of returning None silently.
def find_account(accounts, acct_id):
    for a in accounts:
        if a["id"] == acct_id:
            return a
    raise LookupError(f"no account with id={acct_id}")  # explicit failure, with context

# Failure path: Sam has 40, asks for 100 — `except` catches the raised ValueError.
try:
    withdraw({"id": 3, "owner": "Sam", "balance": 40.0}, 100)
except ValueError as err:
    print("declined:", err)              # declined: insufficient funds: balance 40.0, asked 100

# Happy path: try succeeds, so `else` runs; `finally` runs either way (cleanup/audit).
try:
    found = find_account(accounts, 1)
except LookupError as err:
    print("not found:", err)
else:
    print("found:", found["owner"])      # found: Ada
finally:
    print("lookup attempt done")         # always runs

# Failure path: id 99 isn't present, so `except` catches the LookupError.
try:
    find_account(accounts, 99)
except LookupError as err:
    print("handled:", err)               # handled: no account with id=99

## Next: Module 2

We now have one account as a dict, and we can loop, branch, compute, and fail clearly over a few of them.

**Next we'll store many of these accounts in files** — JSON and CSV — so the bank survives a restart. Same six-field account shape, just persisted and loaded back.